In [1]:
import numpy as np

In [2]:
t1 = np.array([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=np.float32)
t2 = np.array([0.1, 0.5, 1.2, 2.0, 3.5], dtype=np.float32)
t3 = np.array([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=np.float32)
t4 = np.array([5.0, 5.0, 5.0], dtype=np.float32)
t5 = np.array([1e-9, 2e-9, -1e-9], dtype=np.float32)

In [3]:
def calculate_scale_zero_point(tensor, q_min=-128, q_max=127):
    """
    Calculate the scale and zero point for affine INT8 quantization.
    """

    # Find the minimum and maximum values
    x_min = np.min(tensor)
    x_max = np.max(tensor)

    # Handle constant tensor (avoid division by zero)
    if x_max == x_min:
        scale = 1.0
        zero_point = 0
        return scale, zero_point

    # Calculate scale
    scale = (x_max - x_min) / (q_max - q_min)

    # Handle extremely small scale values
    if scale < 1e-12:
        scale = 1e-12

    # Calculate zero point
    zero_point = round(q_min - (x_min / scale))

    # Clip zero point to INT8 range
    zero_point = int(np.clip(zero_point, q_min, q_max))

    return scale, zero_point

In [4]:
def quantize_tensor(tensor, scale, zero_point):
    quantized = np.round(tensor / scale) + zero_point
    quantized = np.clip(quantized, -128, 127)
    return quantized.astype(np.int8)

In [5]:
def dequantize_tensor(quantized_tensor, scale, zero_point):
    return (quantized_tensor.astype(np.float32) - zero_point) * scale

In [6]:
tensors = {
    "Tensor 1": t1,
    "Tensor 2": t2,
    "Tensor 3": t3,
    "Tensor 4": t4,
    "Tensor 5": t5
}

In [7]:
for name, tensor in tensors.items():

    scale, zero_point = calculate_scale_zero_point(tensor)

    quantized = quantize_tensor(tensor, scale, zero_point)

    dequantized = dequantize_tensor(quantized, scale, zero_point)

    mae = np.mean(np.abs(tensor - dequantized))

    print("=" * 50)
    print(name)
    print("=" * 50)

    print("Tensor values:")
    print(tensor)

    print("\nTensor min:", np.min(tensor))
    print("Tensor max:", np.max(tensor))

    print("\nScale:", scale)
    print("Zero Point:", zero_point)

    print("\nQuantized Tensor:")
    print(quantized)

    print("\nDequantized Tensor:")
    print(dequantized)

    print("\nMean Absolute Error (MAE):", mae)
    print()

Tensor 1
Tensor values:
[-1.5 -0.8  0.   0.9  2.3]

Tensor min: -1.5
Tensor max: 2.3

Scale: 0.01490196
Zero Point: -27

Quantized Tensor:
[-128  -81  -27   33  127]

Dequantized Tensor:
[-1.505098   -0.80470586  0.          0.8941176   2.2949018 ]

Mean Absolute Error (MAE): 0.004156864

Tensor 2
Tensor values:
[0.1 0.5 1.2 2.  3.5]

Tensor min: 0.1
Tensor max: 3.5

Scale: 0.013333334
Zero Point: -128

Quantized Tensor:
[-120  -90  -38   22  127]

Dequantized Tensor:
[0.10666667 0.50666666 1.2        2.         3.4       ]

Mean Absolute Error (MAE): 0.022666646

Tensor 3
Tensor values:
[-3.  -2.1 -1.4 -0.6 -0.1]

Tensor min: -3.0
Tensor max: -0.1

Scale: 0.011372549
Zero Point: 127

Quantized Tensor:
[-128  -58    4   74  118]

Dequantized Tensor:
[-2.9        -2.1039217  -1.3988236  -0.6027451  -0.10235295]

Mean Absolute Error (MAE): 0.022039209

Tensor 4
Tensor values:
[5. 5. 5.]

Tensor min: 5.0
Tensor max: 5.0

Scale: 1.0
Zero Point: 0

Quantized Tensor:
[5 5 5]

Dequantized Ten